In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import anndata as ad
import numpy as np
import os
import pandas as pd
from sccnasim.utils.gcna import load_cnas

In [3]:
cna_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/data/cna_profile.tsv'
in_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/rs_benchmark/baf/gene/base/pp'
out_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/rs_benchmark/baf/gene/base/gw-gain/pp'

In [4]:
data_ids = ["seed_normal", "rs_normal", "scrs_normal", "rs_tumor", "scrs_tumor"]

# Prepare Data

## Subset Genes in CN-neutral Regions

In [5]:
os.makedirs(out_dir, exist_ok = True)

In [6]:
cna = load_cnas(cna_fn)
cna

,chrom,start,end,clone,cn_ale0,cn_ale1,region
0,1,123400001,248956422,tumor,1,2,1:123400001-248956422
1,4,50000001,190214555,tumor,0,1,4:50000001-190214555
2,8,1,45200000,tumor,0,1,8:1-45200000
3,8,45200001,145138636,tumor,1,2,8:45200001-145138636
4,13,17700001,114364328,tumor,2,0,13:17700001-114364328
5,17,1,25100000,tumor,2,0,17:1-25100000


In [7]:
cna = cna.loc[cna['cn_ale0'] + cna['cn_ale1'] > 2].copy()
cna

,chrom,start,end,clone,cn_ale0,cn_ale1,region
0,1,123400001,248956422,tumor,1,2,1:123400001-248956422
3,8,45200001,145138636,tumor,1,2,8:45200001-145138636


In [8]:
adata = ad.read_h5ad(os.path.join(in_dir, "rs_normal.h5ad"))
adata

AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [9]:
def is_cna_feature(x, df):
    chrom, start, end = x["chrom"], x["start"], x["end"]
    d = df[(df["chrom"] == chrom) & (df["start"] <= end) & (df["end"] >= start)]
    return(d.shape[0] > 0)

bool_is_cna = adata.var.apply(is_cna_feature, axis = 1, df = cna)
cna_genes = adata.var["feature"][bool_is_cna].to_numpy()

cna_genes.shape

(2303,)

In [10]:
for name in data_ids:
    print("\nstart processing '%s' ..." % name)
    fn = os.path.join(in_dir, "%s.h5ad" % name)
        
    print("\nload adata ...")
    adata = ad.read_h5ad(fn)
    print(adata)

    print("\nsubset adata by CNA genes ...")
    adata = adata[:, adata.var["feature"].isin(cna_genes)].copy()
    print(adata)

    print("\nsave adata ...")
    fn = os.path.join(out_dir, "%s.h5ad" % name)
    adata.write_h5ad(fn, compression = 'gzip')
    
    print("\n------\n")


start processing 'seed_normal' ...

load adata ...
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

subset adata by CNA genes ...
AnnData object with n_obs × n_vars = 600 × 2303
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

save adata ...

------


start processing 'rs_normal' ...

load adata ...
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

subset adata by CNA genes ...
AnnData object with n_obs × n_vars = 600 × 2303
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

save adata ...

------


start processing 'scrs_normal' ...

load adata ...
AnnData object with n_obs × n_vars = 600 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start',